# Manejo de Errores y Context Managers en Python

El manejo de errores es crucial para construir aplicaciones robustas. Python utiliza un sistema de excepciones para gestionar errores en tiempo de ejecución. Además, los Context Managers permiten una gestión eficiente de recursos (como archivos o conexiones) garantizando su liberación.

## 1. El bloque `try` / `except`

Permite capturar errores para evitar que el programa se detenga abruptamente. Es una buena práctica capturar excepciones específicas en lugar de usar un `except` genérico.

In [ ]:
def division_segura(a, b):
    try:
        resultado = a / b
        print(f"Resultado: {resultado}")
    except ZeroDivisionError:
        print("Error: No se puede dividir por cero.")
    except TypeError as e:
        print(f"Error de tipo: {e}")
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")

division_segura(10, 2)
division_segura(10, 0)
division_segura(10, "2")

## 2. Bloques `else` y `finally`

- **`else`**: Se ejecuta si y solo si no ocurrió ninguna excepción en el bloque `try`.
- **`finally`**: Se ejecuta siempre, ocurra o no una excepción. Ideal para tareas de limpieza.

In [ ]:
try:
    f = open("test_error.txt", "w")
    f.write("Escribiendo datos...")
except IOError:
    print("Error al acceder al archivo.")
else:
    print("Escritura completada con éxito.")
finally:
    f.close()
    print("Archivo cerrado.")

## 3. Lanzar Excepciones (`raise`)

Puedes forzar una excepción si se cumple una condición lógica que invalida el proceso.

In [ ]:
def validar_nivel(nivel):
    if nivel < 0:
        raise ValueError(f"El nivel {nivel} no puede ser negativo")
    return True

try:
    validar_nivel(-1)
except ValueError as e:
    print(f"Validación fallida: {e}")

## 4. Excepciones Personalizadas

Para crear tus propios errores, debes heredar de la clase base `Exception`.

In [ ]:
class SaldoInsuficienteError(Exception):
    """Excepción lanzada cuando un usuario intenta retirar más dinero del disponible."""
    def __init__(self, saldo, intento):
        self.saldo = saldo
        self.intento = intento
        super().__init__(f"Intento de retirar {intento} con saldo de {saldo}")

def retirar_dinero(saldo, cantidad):
    if cantidad > saldo:
        raise SaldoInsuficienteError(saldo, cantidad)
    return saldo - cantidad

try:
    retirar_dinero(100, 150)
except SaldoInsuficienteError as e:
    print(f"Error de transacción: {e}")

## 5. Context Managers y la sentencia `with`

Los Context Managers simplifican el manejo de recursos. El ejemplo más común es la apertura de archivos, donde `with` garantiza que el archivo se cierre incluso si ocurre un error.

In [ ]:
with open("data.txt", "w") as f:
    f.write("Uso de Context Manager")
# El archivo se cierra automáticamente aquí

### Crear un Context Manager (Clase)

Implementando los métodos mágicos `__enter__` y `__exit__`.

In [ ]:
class GestionConexion:
    def __init__(self, db_name):
        self.db_name = db_name

    def __enter__(self):
        print(f"Conectando a {self.db_name}...")
        return self # Este objeto se asignará a la variable tras 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Cerrando conexión a {self.db_name}.")
        if exc_type:
            print(f"Se produjo un error: {exc_val}")
        return False # False propaga la excepción, True la silencia

with GestionConexion("Produccion") as db:
    print("Realizando consultas...")
    # raise RuntimeError("Fallo de red")

### Crear un Context Manager (@contextmanager)

Usando el decorador de la librería estándar `contextlib`.

In [ ]:
from contextlib import contextmanager

@contextmanager
def etiqueta_html(etiqueta):
    print(f"<{etiqueta}>")
    try:
        yield
    finally:
        print(f"</{etiqueta}>")

with etiqueta_html("div"):
    with etiqueta_html("p"):
        print("Contenido del párrafo")